# A8 — Retrieval-Augmented Generation with Multiple Embedding Models

**Pipeline:** Data → Chunking → Embeddings → Retrieval → LLM → Output

This notebook builds a RAG system over a knowledge base describing *Unopsis*, our team's
notification-aggregation project, and evaluates **3 embedding models × 3 chunking strategies =
9 configurations** against a fixed generation model.

| Component | Choice |
|---|---|
| Generation model (constant) | `granite3.3:2b` via Ollama (IBM Granite-3.3-2B-Instruct) — evaluated in A6 and A7 Part 1, best of the Medium 2–4B band |
| Small embedding | `all-MiniLM-L6-v2` — 384 dimensions |
| Medium embedding | `all-mpnet-base-v2` — 768 dimensions |
| Large embedding | `bge-large-en-v1.5` — 1024 dimensions |

Granite-3.3-2B is the model our team selected in A6 (0.692 lane accuracy on the triage task, the best below 4B, tied with the larger Phi-4-mini) and A7 Part 1 (9.76/10, 2nd of 15 models), and the one the Unopsis app runs. Ollama serves a 4-bit (Q4_K_M) build of the same weights that A6/A7 ran in full precision through `transformers`.

The generation model **never changes** between configurations. Only the embedding model and the
chunking strategy vary, so any difference in output is attributable to those two components.

### How this notebook is organised

The implementation lives in `ragkit.py`, which this notebook imports. Each section prints the
source of the component it is about, so the code is visible here, but there is only one copy of
it — the code shown is the code that runs. Every table below is computed by the cell above it
during this execution; nothing is loaded from a previous run.

In [1]:
import inspect, json, statistics as st, time
from collections import defaultdict
import numpy as np
from sentence_transformers import SentenceTransformer

import ragkit as rk

show = lambda *fns: print("\n\n".join(inspect.getsource(f) for f in fns))

print("generation model (held constant):", rk.GEN_MODEL)
print("embedding models under test:")
for size, name, dim in rk.EMBEDDERS:
    print(f"   {size:<7} {name:<48} {dim} dimensions")

generation model (held constant): granite3.3:2b
embedding models under test:
   small   sentence-transformers/all-MiniLM-L6-v2           384 dimensions
   medium  sentence-transformers/all-mpnet-base-v2          768 dimensions
   large   BAAI/bge-large-en-v1.5                           1024 dimensions


---
## Step 1.1 — Define the knowledge base

The corpus is product and operations documentation for **Unopsis**, a notification aggregation
service: what it is, its connectors, pricing, the filtering engine, privacy and retention,
support and service levels, the technical architecture, and onboarding.

It is authored as eight Markdown files under `corpus/`. A **paragraph** is the text between two
blank lines — that is how the files are written, and it is the unit every chunking strategy
below is built from.

In [2]:
show(rk.load_documents)

docs = rk.load_documents()
n_paras = sum(len(d["paragraphs"]) for d in docs)
n_words = sum(len(p.split()) for d in docs for p in d["paragraphs"])

print(f"{'document':<20} {'title':<34} {'paras':>5} {'words':>6}")
print("-" * 70)
for d in docs:
    w = sum(len(p.split()) for p in d["paragraphs"])
    print(f"{d['id']:<20} {d['title']:<34} {len(d['paragraphs']):>5} {w:>6}")
print("-" * 70)
print(f"{'TOTAL':<20} {'':<34} {n_paras:>5} {n_words:>6}")

para_words = [len(p.split()) for d in docs for p in d["paragraphs"]]
print(f"\nparagraph length: min {min(para_words)}, max {max(para_words)}, "
      f"mean {st.mean(para_words):.0f} words")
print(f"requirements -- >=20 paragraphs: {n_paras >= 20} | "
      f"2,000-5,000 words: {2000 <= n_words <= 5000} | "
      f"every paragraph 80-200 words: {min(para_words) >= 80 and max(para_words) <= 200}")

def load_documents(corpus_dir=CORPUS_DIR):
    """Each file is one document: a title plus a list of paragraphs.

    A paragraph is defined as text between blank lines, which is how the source files are
    authored. This is the unit every chunking strategy below builds from.
    """
    docs = []
    for p in sorted(pathlib.Path(corpus_dir).glob("*.md")):
        text = p.read_text(encoding="utf-8")   # explicit: Windows would otherwise use cp1252
        title = next((l.lstrip("# ").strip() for l in text.splitlines() if l.startswith("#")), p.stem)
        body = re.sub(r"^#.*$", "", text, flags=re.M).strip()
        paras = [re.sub(r"\s+", " ", x).strip() for x in re.split(r"\n\s*\n", body) if x.strip()]
        docs.append({"id": p.stem, "title": title, "paragraphs": paras})
    return docs

document             title                              paras  words
----------------------------------------------------------------------
01-overview          What Unopsis Is                  

In [3]:
# One paragraph, so the texture of the source data is visible.
print(docs[2]["title"], "/ paragraph 1\n")
print(docs[2]["paragraphs"][0])

Plans and Pricing / paragraph 1

Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per day. The Pro plan costs eight dollars per month, allows fifteen connectors, and processes up to five thousand notifications per day. The Team plan costs twenty dollars per user per month, allows unlimited connectors, and processes up to fifty thousand notifications per day across the whole team. All three plans include the full filtering engine, the unified inbox, mobile push, and the web application. The paid plans are distinguished by capacity, retention, and support rather than by withholding core features.


---
## Step 1.2 — Text representation and chunking

Three strategies, all built from the paragraph unit defined above. Two rules hold across all
three:

* **No sentence is ever split.** The fixed-length strategy packs *whole sentences* up to a word
  budget instead of cutting at an exact offset. A chunk that ends mid-clause both retrieves
  worse and reads worse once it is pasted into a prompt.
* **No chunk ever spans two documents**, so unrelated topics are never merged into one chunk.

### 1. Fixed-length
Accumulate sentences until the next one would exceed a 120-word budget, then start a new chunk.

### 2. Overlapping paragraph
A sliding window of two adjacent paragraphs with stride one: `P1+P2`, `P2+P3`, `P3+P4`, … Every
paragraph except the first and last of a document therefore appears in exactly two chunks.
**Overlap is implemented at paragraph granularity**, which guarantees each chunk is a whole
number of complete thoughts rather than an arbitrary token window.

### 3. Hybrid / strategic
My own strategy, combining two ideas:

* **Title augmentation.** Every chunk is prefixed with its document title. A paragraph that
  says "eight dollars per month" carries no signal about *what* costs eight dollars; prefixing
  "Plans and Pricing" puts the topic inside the embedded text where the retriever can use it.
  This is the main bet of the strategy.
* **Adaptive merge.** Paragraphs shorter than 95 words are merged forward, so the index holds
  no stub chunks that match weakly against everything. A trailing remainder too short to stand
  alone is appended to the previous chunk of the same document.

In [4]:
show(rk.sentences, rk.chunk_fixed, rk.chunk_overlap, rk.chunk_hybrid, rk._chunk)

def sentences(text):
    """Split on sentence terminators followed by a capital. Keeps the terminator attached."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", text) if s.strip()]


def chunk_fixed(docs, budget=120):
    """1. Fixed-length: pack whole sentences up to a `budget` word limit, never splitting one.

    Chunks do not cross document boundaries, so unrelated topics are never merged.
    """
    out = []
    for d in docs:
        sents, cur, n = [s for p in d["paragraphs"] for s in sentences(p)], [], 0
        for s in sents:
            w = len(s.split())
            if cur and n + w > budget:
                out.append(_chunk(d, " ".join(cur), "fixed"))
                cur, n = [], 0
            cur.append(s); n += w
        if cur:
            out.append(_chunk(d, " ".join(cur), "fixed"))
    return out


def chunk_overlap(docs):
    """2. Overlapping paragraph: a sliding window of two adjacent paragraphs, stride one.

    Chunk 1 = P1+P2, chunk 2 = P

In [5]:
import re
chunksets = {name: fn(docs) for name, fn in rk.STRATEGIES.items()}

print(f"{'strategy':<10} {'chunks':>6} {'min':>5} {'max':>5} {'mean':>6}   sentence-safe")
print("-" * 58)
for name, ch in chunksets.items():
    w = [c["words"] for c in ch]
    safe = all(re.search(r'[.!?]["\')]?$', c["text"].strip()) for c in ch)
    print(f"{name:<10} {len(ch):>6} {min(w):>5} {max(w):>5} {st.mean(w):>6.0f}   {safe}")

print("\nSame source paragraph as it appears under each strategy:\n")
for name, ch in chunksets.items():
    hit = next(c for c in ch if "eight dollars per month" in c["text"])
    print(f"[{name}] {hit['words']} words, doc={hit['doc_id']}")
    print(f"   {hit['text'][:190]}...\n")

strategy   chunks   min   max   mean   sentence-safe
----------------------------------------------------------
fixed          32    36   120     97   True
overlap        24   176   210    194   True
hybrid         21    97   199    134   True

Same source paragraph as it appears under each strategy:

[fixed] 100 words, doc=03-pricing
   Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per day. The Pro plan costs eight dollars per month, allows fi...

[overlap] 210 words, doc=03-pricing
   Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per day. The Pro plan costs eight dollars per month, allows fi...

[hybrid] 100 words, doc=03-pricing
   Plans and Pricing. Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per day. The Pro plan costs eight dollars p...

---
## Step 1.3 — Embedding pipeline

Each chunk is encoded into a dense vector and stored in a searchable structure. Vectors are
**L2-normalised at encode time**, which makes cosine similarity a plain dot product, so the
entire search is one matrix–vector multiply.

At this corpus size (21–32 chunks) exact search is faster than building an ANN index *and* has
no recall penalty, so there is no approximation trade-off to report. The same class would need
replacing with FAISS or a hosted vector database at the scale discussed in Part 4.3.

In [6]:
show(rk.VectorStore)

embedders = {}
for size, name, dim in rk.EMBEDDERS:
    t0 = time.perf_counter()
    embedders[size] = {"name": name, "dim": dim, "model": SentenceTransformer(name)}
    embedders[size]["load_sec"] = time.perf_counter() - t0
    print(f"loaded {size:<7} {name:<48} in {embedders[size]['load_sec']:.1f}s")

class VectorStore:
    """Normalised embedding matrix + cosine similarity search.

    Cosine similarity on L2-normalised vectors is a dot product, so the whole search is one
    matrix-vector multiply. At this corpus size that is faster than any index structure and
    exact, so there is no recall/latency trade-off to report.
    """

    def __init__(self, model, chunks, batch=32):
        self.model, self.chunks = model, chunks
        t0 = time.perf_counter()
        self.M = model.encode([c["text"] for c in chunks], batch_size=batch,
                              normalize_embeddings=True, show_progress_bar=False)
        self.index_sec = time.perf_counter() - t0
        self.dim = int(self.M.shape[1])

    def search(self, query, k=3):
        t0 = time.perf_counter()
        q = self.model.encode([query], normalize_embeddings=True)[0]
        t_embed = time.perf_counter() - t0
        t1 = time.perf_counter()
        sims = self.M @ q
        order = np.argsort(-sims)[:k]
      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

loaded small   sentence-transformers/all-MiniLM-L6-v2           in 1.3s


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

... (21 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

loaded large   BAAI/bge-large-en-v1.5                           in 96.2s


In [7]:
# Build all nine indexes once and reuse them for every experiment below.
stores = {}
print(f"{'embedding':<8} {'strategy':<9} {'dim':>5} {'chunks':>7} {'index build':>12}")
print("-" * 48)
for size in ("small", "medium", "large"):
    for strat in ("fixed", "overlap", "hybrid"):
        s = rk.VectorStore(embedders[size]["model"], chunksets[strat])
        stores[(size, strat)] = s
        print(f"{size:<8} {strat:<9} {s.dim:>5} {len(s.chunks):>7} {s.index_sec:>11.2f}s")

embedding strategy    dim  chunks  index build
------------------------------------------------


small    fixed       384      32        1.48s


small    overlap     384      24        1.11s


small    hybrid      384      21        0.94s


medium   fixed       768      32       13.95s


medium   overlap     768      24       14.42s


medium   hybrid      768      21        9.46s


large    fixed      1024      32       21.04s


large    overlap    1024      24       27.36s


large    hybrid     1024      21       28.84s


---
## Step 1.4 — Retrieval system

A query is encoded with the *same* model that built the index, then scored against every chunk
by cosine similarity, and the top *k* are returned. Each hit carries its similarity score, its
rank, and the document it came from, which is what makes the failure analysis in Part 3
possible.

In [8]:
show(rk.VectorStore.search)

demo_q = "How much does the Pro plan cost per month?"
hits, t_embed, t_search = stores[("medium", "hybrid")].search(demo_q, k=3)
print(f"query: {demo_q!r}")
print(f"query embedding {t_embed*1000:.1f} ms | similarity search {t_search*1000:.2f} ms\n")
for h in hits:
    print(f"  rank {h['rank']}  score {h['score']:.4f}  {h['doc_id']} ({h['doc_title']})")
    print(f"      {h['text'][:150]}...\n")

    def search(self, query, k=3):
        t0 = time.perf_counter()
        q = self.model.encode([query], normalize_embeddings=True)[0]
        t_embed = time.perf_counter() - t0
        t1 = time.perf_counter()
        sims = self.M @ q
        order = np.argsort(-sims)[:k]
        t_search = time.perf_counter() - t1
        hits = [{**self.chunks[i], "score": float(sims[i]), "rank": r + 1}
                for r, i in enumerate(order)]
        return hits, t_embed, t_search



query: 'How much does the Pro plan cost per month?'
query embedding 2309.1 ms | similarity search 20.39 ms

  rank 1  score 0.7246  03-pricing (Plans and Pricing)
      Plans and Pricing. Annual billing is available on Pro and Team and is discounted by two months, so an annual Pro subscription costs eighty dollars per...

  rank 2  score 0.5592  06-support (Support and Service Levels)
      Support and Service Levels. The Team plan carries a ninety nine point nine percent monthly uptime commitment. Uptime is measured against the notificat...

  rank 3  score 0.4875  03-pricing (Plans and Pricing)
      Plans and Pricing. Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per...



---
## Step 1.5 — Generation pipeline

The prompt has the three parts the assignment specifies:

1. a **system instruction**, which also tells the model to answer `NOT IN CONTEXT` when the
   retrieved text does not contain the answer — this is what makes an abstention
   distinguishable from a hallucination in the results;
2. the **retrieved context**, each chunk labelled with its rank and source document title;
3. the **user query**.

Generation uses `temperature=0` and a fixed seed, so differences between configurations come
from retrieval rather than from sampling noise.

In [9]:
print(rk.SYSTEM)
print()
show(rk.build_prompt, rk.generate)

You are a support assistant for Unopsis. Answer the question using only the context provided. If the context does not contain the answer, reply exactly: NOT IN CONTEXT. Answer in one or two sentences and state any number exactly as the context gives it.

def build_prompt(query, hits):
    ctx = "\n\n".join(f"[{h['rank']}] ({h['doc_title']}) {h['text']}" for h in hits)
    return f"{SYSTEM}\n\nContext:\n{ctx}\n\nQuestion: {query}\nAnswer:"


def generate(prompt, model=GEN_MODEL, num_predict=160, seed=42, timeout=180):
    body = json.dumps({"model": model, "prompt": prompt, "stream": False,
                       "options": {"temperature": 0, "seed": seed, "num_predict": num_predict}}).encode()
    req = urllib.request.Request(OLLAMA, data=body, headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        d = json.loads(r.read())
    return {"text": d.get("response", "").strip(), "sec": time.perf_counter()

In [10]:
prompt = rk.build_prompt(demo_q, hits)
print("----- PROMPT SENT TO THE MODEL -----")
print(prompt[:1100], "\n   ...\n")
out = rk.generate(prompt)
print("----- ANSWER -----")
print(out["text"])
print(f"\n({out['sec']:.2f}s, {out['eval_count']} tokens generated, finish={out['done_reason']})")

----- PROMPT SENT TO THE MODEL -----
You are a support assistant for Unopsis. Answer the question using only the context provided. If the context does not contain the answer, reply exactly: NOT IN CONTEXT. Answer in one or two sentences and state any number exactly as the context gives it.

Context:
[1] (Plans and Pricing) Plans and Pricing. Annual billing is available on Pro and Team and is discounted by two months, so an annual Pro subscription costs eighty dollars per year rather than ninety six. Students with a verified academic email address receive fifty percent off the Pro plan for twelve months, which brings it to four dollars per month. The student discount may be renewed once by reverifying, giving a maximum of twenty four discounted months. Team plans are billed per seat, and seats may be added at any time. Adding a seat mid cycle is prorated to the day; removing a seat takes effect at the end of the current billing period rather than immediately.

[2] (Support and Service L

----- ANSWER -----
The Pro plan costs eight dollars per month.

(27.12s, 11 tokens generated, finish=stop)


---
# Part 2 — System experiments

## Step 2.1 — Test queries

Fourteen queries in two sets, and the distinction drives the whole analysis.

**Direct (Q1–Q7)** phrase the question roughly the way the corpus phrases the answer.

**Hard (H1–H7)** were written after the direct set turned out to *saturate* in our first runs — on eight
topically distinct documents, a question that reuses the corpus's own wording is found by every
embedding model at top-3, so every configuration scored full marks and there was nothing to
compare. Each hard query uses the wording a real user would reach for **and** competes with a
near neighbour elsewhere in the corpus:

| Query | What makes it hard |
|---|---|
| H1 | asks about refunds without ever using the word "refund" |
| H2 | "how long do I have" collides with the refund window and three separate retention windows |
| H3 | requires knowing which *plan* gates a feature — the answer joins two documents |
| H4 | asks for support channels, which the corpus states right next to the channels it does *not* offer |
| H5 | "seven days" appears in both the connector-expiry and the data-retention documents |
| H6 | describes a symptom ("goes silent overnight") instead of naming the feature |
| H7 | "five" appears in the corpus as five retries, five megabytes, and five business days |

Each query carries the document that actually answers it (`gold_doc`) and the accepted surface
forms of the correct fact. The corpus spells numbers out while the queries use digits, so
retrieval cannot succeed on lexical overlap alone — grading accepts either form.

With the Granite generator both sets produce failures, of different kinds. On the direct set the model sometimes picks the wrong figure out of a correct chunk. On the hard set the failures come from ambiguous wording and retrieval collisions. Part 3 has the details.

In [11]:
print(f"{'id':<4} {'set':<7} {'gold doc':<15} query")
print("-" * 92)
for q in rk.QUERIES:
    print(f"{q['id']:<4} {q['difficulty']:<7} {q['gold_doc']:<15} {q['text']}")
print(f"\n{len(rk.QUERIES)} queries ({len(rk.DIRECT_QUERIES)} direct + {len(rk.HARD_QUERIES)} hard)")

id   set     gold doc        query
--------------------------------------------------------------------------------------------
Q1   direct  03-pricing      How much does the Pro plan cost per month?
Q2   direct  05-privacy      How long are notifications kept on the free plan?
Q3   direct  04-filtering    What is the default priority threshold for immediate delivery?
Q4   direct  06-support      What uptime does the Team plan commit to each month?
Q5   direct  07-architecture How many API requests per minute can a Pro account make?
Q6   direct  02-connectors   Which connector delivers events by webhook instead of polling?
Q7   direct  03-pricing      How long is the refund window on a paid plan?
H1   hard    03-pricing      If I cancel my subscription, can I get my money back?
H2   hard    06-support      How long do I have to request a service credit after an outage?
H3   hard    05-privacy      Which plan do I need if my organisation requires single sign on?
H4   hard    06-support 

### The grading rubric

Both scores are mechanical, so they can be checked rather than trusted.

**Retrieved context quality (0–5)** — *did retrieval put the answer in front of the model?*

| | |
|---|---|
| 5 | a retrieved chunk actually contains the answer |
| 3 | the right document was retrieved, but not the part containing the answer |
| 0 | the right document was not retrieved at all |

**Answer quality (0–5)** — *did the model then use it?*

| | |
|---|---|
| 5 | states the correct fact |
| 1 | on topic, but never states the fact |
| 0 | abstained (`NOT IN CONTEXT`), or answered with a contradicting fact |

Separating the two is the point: a 5/1 row is a *generation* failure, a 0/0 row is a
*retrieval* failure, and they need different fixes. We also record **MRR** over the rank at
which the answer first appears, because once several configurations all reach context quality
5.0, the rank is the only thing still distinguishing them.

One deliberate choice: an answer that states the correct fact **and** adds a neighbouring one
("Pro is eight dollars, Team is twenty") scores 5. An earlier version docked a point for that,
which measured verbosity rather than accuracy. A conflicting figure is still recorded in
`also_states_conflicting` so it stays visible in the data.

In [12]:
show(rk.grade_context, rk.grade_answer)

def grade_context(q, hits):
    """0-5 for retrieved context, plus the rank at which the answer first appears.

    5 = a retrieved chunk actually contains the answer
    3 = the right document was retrieved, but not the part containing the answer
    0 = the right document was not retrieved at all
    """
    rank = next((h["rank"] for h in hits
                 if any(g in _norm(h["text"]) for g in q["gold"])), None)
    from_gold_doc = any(h["doc_id"] == q["gold_doc"] for h in hits)
    score = 5 if rank else (3 if from_gold_doc else 0)
    return score, rank, from_gold_doc


def grade_answer(q, answer):
    """0-5 for the generated answer.

    5 = states the correct fact
    1 = attempted an answer, on topic, but never states the fact
    0 = abstained (NOT IN CONTEXT), returned nothing, or answered with a contradicting fact

    Revised after the first run: the earlier version docked a point when the answer stated the
    correct fact AND mentioned a different figure. That is wro

## Step 2.2 — Evaluation table: all 9 configurations

Running 9 configurations × 14 queries live. Roughly five minutes, almost all of it
generation.

In [13]:
K = 3
grid = []
for size in ("small", "medium", "large"):
    for strat in ("fixed", "overlap", "hybrid"):
        store = stores[(size, strat)]
        rows = rk.run_config(store, k=K)
        by_diff = {}
        for d in ("direct", "hard"):
            sub = [r for r in rows if r["difficulty"] == d]
            by_diff[d] = {
                "ctx": st.mean(r["context_quality"] for r in sub),
                "ans": st.mean(r["answer_quality"] for r in sub),
                "mrr": st.mean((1 / r["answer_rank"]) if r["answer_rank"] else 0 for r in sub),
                "recall": sum(1 for r in sub if r["gold_doc_retrieved"]), "n": len(sub),
            }
        grid.append({
            "embed_size": size, "embed_model": embedders[size]["name"], "dim": store.dim,
            "strategy": strat, "n_chunks": len(store.chunks),
            "mean_chunk_words": st.mean(c["words"] for c in store.chunks),
            "index_sec": store.index_sec, "rows": rows, "by_difficulty": by_diff,
            "ctx": st.mean(r["context_quality"] for r in rows),
            "ans": st.mean(r["answer_quality"] for r in rows),
            "mrr": st.mean((1 / r["answer_rank"]) if r["answer_rank"] else 0 for r in rows),
            "retrieval_ms": st.mean((r["latency"]["embed_query"] + r["latency"]["search"]) * 1000
                                    for r in rows),
            "gen_sec": st.mean(r["latency"]["generate"] for r in rows),
            "total_sec": st.mean(r["latency"]["total"] for r in rows),
            "gold_doc_recall": sum(1 for r in rows if r["gold_doc_retrieved"]),
        })
        c = grid[-1]
        print(f"  {size:<7} {strat:<8} ctx {c['ctx']:.2f} ans {c['ans']:.2f} "
              f"mrr {c['mrr']:.3f} | {c['total_sec']:.2f}s/q", flush=True)
print(f"\n{len(grid)} configurations complete")

  small   fixed    ctx 5.00 ans 4.64 mrr 0.881 | 9.92s/q


  small   overlap  ctx 4.64 ans 4.64 mrr 0.929 | 15.40s/q


  small   hybrid   ctx 5.00 ans 4.71 mrr 0.917 | 14.11s/q


  medium  fixed    ctx 4.86 ans 4.36 mrr 0.821 | 10.53s/q


  medium  overlap  ctx 5.00 ans 5.00 mrr 0.881 | 14.31s/q


  medium  hybrid   ctx 5.00 ans 4.64 mrr 0.786 | 11.95s/q


  large   fixed    ctx 5.00 ans 4.36 mrr 0.929 | 10.44s/q


  large   overlap  ctx 5.00 ans 4.71 mrr 0.929 | 15.11s/q


  large   hybrid   ctx 5.00 ans 4.71 mrr 0.881 | 13.75s/q



9 configurations complete


In [14]:
hdr = (f"{'embedding':<8} {'dim':>5} {'strategy':<9} {'chunks':>6} {'ctx':>5} {'ans':>5} "
       f"{'MRR':>6} {'recall':>7} {'retr ms':>8} {'gen s':>6} {'total s':>8}")
print(hdr); print("-" * len(hdr))
for c in grid:
    print(f"{c['embed_size']:<8} {c['dim']:>5} {c['strategy']:<9} {c['n_chunks']:>6} "
          f"{c['ctx']:>5.2f} {c['ans']:>5.2f} {c['mrr']:>6.3f} "
          f"{c['gold_doc_recall']:>4}/{len(rk.QUERIES):<2} {c['retrieval_ms']:>8.1f} "
          f"{c['gen_sec']:>6.2f} {c['total_sec']:>8.2f}")
print("-" * len(hdr))
print("\nSplit by query set (context quality / answer quality):")
print(f"{'embedding':<8} {'strategy':<9} {'direct':>14} {'hard':>14}")
print("-" * 48)
for c in grid:
    d, h = c["by_difficulty"]["direct"], c["by_difficulty"]["hard"]
    print(f"{c['embed_size']:<8} {c['strategy']:<9} {d['ctx']:>6.2f} /{d['ans']:>6.2f} "
          f"{h['ctx']:>7.2f} /{h['ans']:>6.2f}")

embedding   dim strategy  chunks   ctx   ans    MRR  recall  retr ms  gen s  total s
------------------------------------------------------------------------------------
small      384 fixed         32  5.00  4.64  0.881   14/14    132.0   9.79     9.92
small      384 overlap       24  4.64  4.64  0.929   13/14     74.5  15.33    15.40
small      384 hybrid        21  5.00  4.71  0.917   14/14    119.1  13.99    14.11
medium     768 fixed         32  4.86  4.36  0.821   14/14    849.7   9.69    10.53
medium     768 overlap       24  5.00  5.00  0.881   14/14    287.3  14.02    14.31
medium     768 hybrid        21  5.00  4.64  0.786   13/14    210.7  11.74    11.95
large     1024 fixed         32  5.00  4.36  0.929   14/14   1354.0   9.09    10.44
large     1024 overlap       24  5.00  4.71  0.929   14/14   1250.9  13.86    15.11
large     1024 hybrid        21  5.00  4.71  0.881   14/14   1459.3  12.29    13.75
--------------------------------------------------------------------------

### Per-query detail, including the retrieved chunks

The assignment requires the **retrieved chunks**, not just the final answers. So this prints every row for the best
and worst configurations by answer quality with the full text of each retrieved chunk, then the
complete 126-row summary table, and then every configuration in turn with the opening of each
retrieved chunk, the answer, both scores, the latency and a note. All of it is saved in `results/notebook_run.json`.

In [15]:
best = max(grid, key=lambda c: (c["ans"], c["mrr"]))
worst = min(grid, key=lambda c: (c["ans"], c["mrr"]))

for tag, c in (("BEST", best), ("WORST", worst)):
    print("=" * 100)
    print(f"{tag}: {c['embed_size']} ({c['dim']}d) + {c['strategy']} | "
          f"ctx {c['ctx']:.2f} ans {c['ans']:.2f} mrr {c['mrr']:.3f}")
    print("=" * 100)
    for r in c["rows"]:
        flag = "" if r["answer_quality"] == 5 else "   <<< NOT FULL MARKS"
        print(f"\n[{r['query_id']} / {r['difficulty']}] {r['query']}")
        print(f"   gold doc {r['gold_doc']} | context {r['context_quality']}/5 | "
              f"answer {r['answer_quality']}/5 ({r['verdict']}) | "
              f"answer first at rank {r['answer_rank']}{flag}")
        print("   RETRIEVED CHUNKS:")
        for h in r["retrieved"]:
            mark = "*" if h["doc_id"] == r["gold_doc"] else " "
            print(f"     {mark}rank {h['rank']} score {h['score']:.4f} [{h['doc_id']}] "
                  f"{h['words']}w")
            print(f"        {h['text'][:260]}")
        print(f"   ANSWER: {r['answer']}")

BEST: medium (768d) + overlap | ctx 5.00 ans 5.00 mrr 0.881

[Q1 / direct] How much does the Pro plan cost per month?
   gold doc 03-pricing | context 5/5 | answer 5/5 (correct) | answer first at rank 2
   RETRIEVED CHUNKS:
      rank 1 score 0.5532 [06-support] 198w
        Support is delivered by email and by an in application contact form. The Free plan receives best effort support with no committed response time, and in practice most free tier questions are answered within three business days. The Pro plan commits to a first r
     *rank 2 score 0.5516 [03-pricing] 210w
        Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two hundred notifications per day. The Pro plan costs eight dollars per month, allows fifteen connectors, and processes up to five thousand notifications per 
     *rank 3 score 0.5073 [03-pricing] 200w
        Annual billing is available on Pro and Team and is discounted by two months, so an annual Pro subs

In [16]:
# Complete 126-row table.
print(f"{'cfg':<18} {'q':<4} {'set':<7} {'ctx':>4} {'ans':>4} {'rank':>5} {'retrieved docs':<34} verdict")
print("-" * 118)
for c in grid:
    cfg = f"{c['embed_size']}/{c['strategy']}"
    for r in c["rows"]:
        rdocs = ",".join(h["doc_id"].split("-")[0] for h in r["retrieved"])
        print(f"{cfg:<18} {r['query_id']:<4} {r['difficulty']:<7} {r['context_quality']:>4} "
              f"{r['answer_quality']:>4} {str(r['answer_rank']):>5} {rdocs:<34} {r['verdict']}")

cfg                q    set      ctx  ans  rank retrieved docs                     verdict
----------------------------------------------------------------------------------------------------------------------
small/fixed        Q1   direct     5    5     3 03,06,03                           correct
small/fixed        Q2   direct     5    5     1 05,06,01                           correct
small/fixed        Q3   direct     5    5     1 04,08,04                           correct
small/fixed        Q4   direct     5    5     2 06,06,01                           correct
small/fixed        Q5   direct     5    0     1 07,06,03                           states a contradicting fact
small/fixed        Q6   direct     5    5     1 02,02,07                           correct
small/fixed        Q7   direct     5    5     1 03,03,06                           correct
small/fixed        H1   hard       5    5     1 03,03,02                           correct
small/fixed        H2   hard       5    5 

### Every configuration: retrieved chunks, answer, scores, latency, notes

The full Step 2.2 table, one block per configuration. `*` marks a chunk from the document that
holds the answer; the score is cosine similarity. The note says in words what the two scores
mean for that row.

In [17]:
def note(r):
    if r["context_quality"] == 0:
        return "retrieval miss: the answer's document is not in the top-k"
    if r["context_quality"] == 3:
        return "right document, wrong chunk: the answer was cut away by the chunk boundary"
    if r["answer_quality"] == 5:
        extra = []
        if r["answer_rank"] and r["answer_rank"] > 1:
            extra.append(f"answer only at rank {r['answer_rank']}")
        if r["also_states_conflicting"]:
            extra.append("also mentions a conflicting figure")
        return "correct" + (" (" + "; ".join(extra) + ")" if extra else "")
    if r["verdict"] == "abstained":
        return "the answer was retrieved, but the model abstained"
    if r["answer_quality"] == 0:
        return "the answer was retrieved, but the model stated a competing figure"
    return "the answer was retrieved, but the model did not state it"

for c in grid:
    print("=" * 110)
    print(f"{c['embed_size'].upper()} ({c['embed_model'].split('/')[-1]}, {c['dim']}d) + "
          f"{c['strategy']} chunking | ctx {c['ctx']:.2f} ans {c['ans']:.2f} mrr {c['mrr']:.3f}")
    print("=" * 110)
    for r in c["rows"]:
        retr_ms = (r["latency"]["embed_query"] + r["latency"]["search"]) * 1000
        print(f"{r['query_id']} [{r['difficulty']}] {r['query']}")
        for h in r["retrieved"]:
            mark = "*" if h["doc_id"] == r["gold_doc"] else " "
            print(f"   {mark}{h['rank']}. {h['score']:.3f} [{h['doc_id']}] {h['text'][:105]}...")
        print(f"   answer : {r['answer'][:170]}")
        print(f"   context {r['context_quality']}/5 | answer {r['answer_quality']}/5 | "
              f"latency {r['latency']['total']:.2f} s (retrieval {retr_ms:.0f} ms) | note: {note(r)}\n")


SMALL (all-MiniLM-L6-v2, 384d) + fixed chunking | ctx 5.00 ans 4.64 mrr 0.881
Q1 [direct] How much does the Pro plan cost per month?
   *1. 0.717 [03-pricing] Annual billing is available on Pro and Team and is discounted by two months, so an annual Pro subscriptio...
    2. 0.584 [06-support] Support is delivered by email and by an in application contact form. The Free plan receives best effort s...
   *3. 0.565 [03-pricing] Unopsis offers three plans. The Free plan costs nothing, allows three connectors, and processes up to two...
   answer : The Pro plan costs eight dollars per month.
   context 5/5 | answer 5/5 | latency 10.39 s (retrieval 354 ms) | note: correct (answer only at rank 3)

Q2 [direct] How long are notifications kept on the free plan?
   *1. 0.717 [05-privacy] Retention is determined by plan. The Free plan retains notifications for seven days, the Pro plan for nin...
    2. 0.615 [06-support] Support is delivered by email and by an in application contact form. The Free

## Step 2.3 — Compare embedding models

Aggregated across the three chunking strategies, so each row is one embedding model's mean
over 3 strategies × 14 queries = 42 queries.

In [18]:
def agg(key):
    out = {}
    for c in grid:
        out.setdefault(c[key], []).append(c)
    return out

print(f"{'embedding':<8} {'dim':>5} {'ctx':>6} {'ans':>6} {'MRR':>7} {'hard ans':>9} "
      f"{'index s':>8} {'retr ms':>8} {'load s':>7}")
print("-" * 76)
for size in ("small", "medium", "large"):
    cs = agg("embed_size")[size]
    print(f"{size:<8} {cs[0]['dim']:>5} "
          f"{st.mean(c['ctx'] for c in cs):>6.2f} {st.mean(c['ans'] for c in cs):>6.2f} "
          f"{st.mean(c['mrr'] for c in cs):>7.3f} "
          f"{st.mean(c['by_difficulty']['hard']['ans'] for c in cs):>9.2f} "
          f"{st.mean(c['index_sec'] for c in cs):>8.2f} "
          f"{st.mean(c['retrieval_ms'] for c in cs):>8.1f} "
          f"{embedders[size]['load_sec']:>7.1f}")

print("\nPer configuration, ranked by answer quality then MRR:")
for i, c in enumerate(sorted(grid, key=lambda c: (-c["ans"], -c["mrr"])), 1):
    print(f"  {i:>2}. {c['embed_size']:<7} ({c['dim']:>4}d) + {c['strategy']:<8} "
          f"ans {c['ans']:.2f}  mrr {c['mrr']:.3f}  ctx {c['ctx']:.2f}")

bysize = {s: st.mean(c["ans"] for c in agg("embed_size")[s]) for s in ("small", "medium", "large")}
print(f"\nmean answer quality by dimension: "
      + "  ".join(f"{k} {v:.2f}" for k, v in bysize.items()))
print(f"larger embeddings monotonically better? "
      f"{bysize['small'] <= bysize['medium'] <= bysize['large']}")

embedding   dim    ctx    ans     MRR  hard ans  index s  retr ms  load s
----------------------------------------------------------------------------
small      384   4.88   4.67   0.909      5.00     1.18    108.5     1.3
medium     768   4.95   4.67   0.829      4.76    12.61    449.3    54.6
large     1024   5.00   4.60   0.913      4.19    25.75   1354.7    96.2

Per configuration, ranked by answer quality then MRR:
   1. medium  ( 768d) + overlap  ans 5.00  mrr 0.881  ctx 5.00
   2. large   (1024d) + overlap  ans 4.71  mrr 0.929  ctx 5.00
   3. small   ( 384d) + hybrid   ans 4.71  mrr 0.917  ctx 5.00
   4. large   (1024d) + hybrid   ans 4.71  mrr 0.881  ctx 5.00
   5. small   ( 384d) + overlap  ans 4.64  mrr 0.929  ctx 4.64
   6. small   ( 384d) + fixed    ans 4.64  mrr 0.881  ctx 5.00
   7. medium  ( 768d) + hybrid   ans 4.64  mrr 0.786  ctx 5.00
   8. large   (1024d) + fixed    ans 4.36  mrr 0.929  ctx 5.00
   9. medium  ( 768d) + fixed    ans 4.36  mrr 0.821  ctx 4.86

mean an

## Step 2.4 — Compare chunking strategies

Aggregated across the three embedding models, so each row is one strategy's mean over
3 models × 14 queries.

In [19]:
print(f"{'strategy':<9} {'chunks':>6} {'mean words':>11} {'ctx':>6} {'ans':>6} {'MRR':>7} "
      f"{'direct ans':>11} {'hard ans':>9} {'index s':>8}")
print("-" * 82)
for strat in ("fixed", "overlap", "hybrid"):
    cs = agg("strategy")[strat]
    print(f"{strat:<9} {cs[0]['n_chunks']:>6} {cs[0]['mean_chunk_words']:>11.0f} "
          f"{st.mean(c['ctx'] for c in cs):>6.2f} {st.mean(c['ans'] for c in cs):>6.2f} "
          f"{st.mean(c['mrr'] for c in cs):>7.3f} "
          f"{st.mean(c['by_difficulty']['direct']['ans'] for c in cs):>11.2f} "
          f"{st.mean(c['by_difficulty']['hard']['ans'] for c in cs):>9.2f} "
          f"{st.mean(c['index_sec'] for c in cs):>8.2f}")

# How much context does each strategy actually push into the prompt at k=3?
print(f"\n{'strategy':<9} {'words in prompt at k=3':>24}")
for strat in ("fixed", "overlap", "hybrid"):
    c = agg("strategy")[strat][0]
    w = st.mean(sum(h["words"] for h in r["retrieved"]) for r in c["rows"])
    print(f"{strat:<9} {w:>24.0f}")

strategy  chunks  mean words    ctx    ans     MRR  direct ans  hard ans  index s
----------------------------------------------------------------------------------
fixed         32          97   4.95   4.45   0.877        4.33      4.57    12.16
overlap       24         194   4.88   4.79   0.913        4.76      4.81    14.30
hybrid        21         134   5.00   4.69   0.861        4.81      4.57    13.08

strategy    words in prompt at k=3
fixed                          294
overlap                        578
hybrid                         417


## Step 2.5 — Data scaling experiment

The configuration is held fixed and the index is grown from 2 documents to all 8. Only queries
whose answer lies inside the current subset are asked, because a question whose answer is not
in the corpus is a test of abstention rather than of retrieval.

What to watch: `off-doc chunks` is how many of the retrieved slots came from a document other
than the one holding the answer — in other words, how much **noise** the retriever pulls in as
the corpus grows.

In [20]:
SCALE_SIZE, SCALE_STRAT = "small", "hybrid"
scaling = []
for n in (2, 4, 6, 8):
    subset = docs[:n]
    chunks = rk.STRATEGIES[SCALE_STRAT](subset)
    store = rk.VectorStore(embedders[SCALE_SIZE]["model"], chunks)
    ids = {d["id"] for d in subset}
    answerable = [q for q in rk.QUERIES if q["gold_doc"] in ids]
    rows = rk.run_config(store, queries=answerable, k=K)
    slots = sum(len(r["retrieved"]) for r in rows)
    offdoc = sum(1 for r in rows for h in r["retrieved"] if h["doc_id"] != r["gold_doc"])
    scaling.append({
        "n_docs": n, "n_paras": sum(len(d["paragraphs"]) for d in subset),
        "n_chunks": len(chunks), "n_queries": len(rows),
        "ctx": st.mean(r["context_quality"] for r in rows),
        "ans": st.mean(r["answer_quality"] for r in rows),
        "mrr": st.mean((1 / r["answer_rank"]) if r["answer_rank"] else 0 for r in rows),
        "offdoc": offdoc, "slots": slots, "noise": offdoc / slots,
        "rows": rows,
    })
    s = scaling[-1]
    print(f"  {n} docs | {s['n_paras']:>2} paras | {s['n_chunks']:>2} chunks | "
          f"{s['n_queries']:>2} queries | ctx {s['ctx']:.2f} ans {s['ans']:.2f} "
          f"mrr {s['mrr']:.3f} | off-doc {offdoc}/{slots} = {s['noise']:.0%}", flush=True)

print(f"\n{'docs':>5} {'chunks':>7} {'queries':>8} {'ctx':>6} {'ans':>6} {'MRR':>7} {'noise':>7}")
print("-" * 50)
for s in scaling:
    print(f"{s['n_docs']:>5} {s['n_chunks']:>7} {s['n_queries']:>8} {s['ctx']:>6.2f} "
          f"{s['ans']:>6.2f} {s['mrr']:>7.3f} {s['noise']:>6.0%}")

  2 docs |  8 paras |  7 chunks |  2 queries | ctx 5.00 ans 5.00 mrr 1.000 | off-doc 0/6 = 0%


  4 docs | 16 paras | 12 chunks |  7 queries | ctx 5.00 ans 5.00 mrr 0.929 | off-doc 3/21 = 14%


  6 docs | 24 paras | 17 chunks | 12 queries | ctx 5.00 ans 5.00 mrr 0.903 | off-doc 13/36 = 36%


  8 docs | 32 paras | 21 chunks | 14 queries | ctx 5.00 ans 5.00 mrr 0.917 | off-doc 18/42 = 43%



 docs  chunks  queries    ctx    ans     MRR   noise
--------------------------------------------------
    2       7        2   5.00   5.00   1.000     0%
    4      12        7   5.00   5.00   0.929    14%
    6      17       12   5.00   5.00   0.903    36%
    8      21       14   5.00   5.00   0.917    43%


---
# Part 3 — Failure analysis and improvement

## Step 3.1 — Failure cases

A row is a failure if retrieval missed (context < 5) or the answer did not state the fact
(answer < 5). These are pulled straight out of the 126 rows above, not constructed
separately.

In [21]:
failures = [dict(r, cfg=f"{c['embed_size']}/{c['strategy']}", dim=c["dim"])
            for c in grid for r in c["rows"]
            if r["context_quality"] < 5 or r["answer_quality"] < 5]

print(f"{len(failures)} failing rows out of {sum(len(c['rows']) for c in grid)}\n")
print(f"{'cfg':<18} {'q':<4} {'ctx':>4} {'ans':>4} {'rank':>5} verdict")
print("-" * 70)
for f in failures:
    print(f"{f['cfg']:<18} {f['query_id']:<4} {f['context_quality']:>4} "
          f"{f['answer_quality']:>4} {str(f['answer_rank']):>5} {f['verdict']}")

# group identical failures so the distinct modes are visible
modes = {}
for f in failures:
    modes.setdefault((f["query_id"], f["context_quality"], f["answer_quality"]), []).append(f["cfg"])
print(f"\n{len(modes)} distinct (query, context, answer) failure patterns:")
for (qid, cq, aq), cfgs in sorted(modes.items()):
    print(f"  {qid}  ctx {cq} ans {aq}  in {len(cfgs)} config(s): {', '.join(cfgs)}")

10 failing rows out of 126



cfg                q     ctx  ans  rank verdict
----------------------------------------------------------------------
small/fixed        Q5      5    0     1 states a contradicting fact
small/overlap      Q2      0    0  None abstained
small/hybrid       Q7      5    1     1 on topic, fact missing
medium/fixed       Q1      3    0  None abstained
medium/fixed       Q7      5    1     1 on topic, fact missing
medium/hybrid      H7      5    0     1 abstained
large/fixed        H1      5    0     1 abstained
large/fixed        H6      5    1     1 on topic, fact missing
large/overlap      H6      5    1     2 on topic, fact missing
large/hybrid       H6      5    1     2 on topic, fact missing

7 distinct (query, context, answer) failure patterns:
  H1  ctx 5 ans 0  in 1 config(s): large/fixed
  H6  ctx 5 ans 1  in 3 config(s): large/fixed, large/overlap, large/hybrid
  H7  ctx 5 ans 0  in 1 config(s): medium/hybrid
  Q1  ctx 3 ans 0  in 1 config(s): medium/fixed
  Q2  ctx 0 ans 0  in 

In [22]:
# At least three failures in full, covering every failure mode, with the retrieved chunks.
def classify(f):
    if f["context_quality"] == 0:
        return ("RETRIEVAL", "the document holding the answer was never retrieved")
    if f["context_quality"] == 3:
        return ("CHUNKING", "the right document was retrieved, but not the chunk with the answer")
    if f["answer_quality"] == 0 and f["verdict"] == "abstained":
        return ("GENERATION", "the answer was in the context and the model abstained anyway")
    if f["answer_quality"] == 0:
        return ("GENERATION", "the model answered with a competing fact from the context")
    return ("GENERATION", "the answer was in the context and the model did not state it")

# One example of each failure mode first, then failures on other queries, so that at
# least three are always shown however few modes a run produces.
picked, seen, seen_q = [], set(), set()
for f in failures:
    if classify(f) not in seen:
        seen.add(classify(f)); seen_q.add(f["query_id"]); picked.append(f)
for f in failures:
    if len(picked) >= 3:
        break
    if f["query_id"] not in seen_q:
        seen_q.add(f["query_id"]); picked.append(f)
for f in failures:
    if len(picked) >= 3:
        break
    if f not in picked:
        picked.append(f)

shown = 0
for f in picked:
    kind = classify(f); shown += 1
    print("=" * 100)
    print(f"FAILURE {shown}  [{kind[0]}]  {kind[1]}")
    print(f"  config: {f['cfg']} ({f['dim']}d) | query {f['query_id']} ({f['difficulty']})")
    print(f"  question : {f['query']}")
    print(f"  gold doc : {f['gold_doc']} | context {f['context_quality']}/5 | "
          f"answer {f['answer_quality']}/5 ({f['verdict']})")
    print("  retrieved:")
    for h in f["retrieved"]:
        mark = "*" if h["doc_id"] == f["gold_doc"] else " "
        print(f"    {mark}rank {h['rank']} score {h['score']:.4f} [{h['doc_id']}] {h['words']}w")
        print(f"       {h['text'][:300]}")
    print(f"  ANSWER   : {f['answer']}")
print("=" * 100)
print(f"\n{shown} failures shown, covering {len(seen)} distinct failure mode(s).")

FAILURE 1  [GENERATION]  the model answered with a competing fact from the context
  config: small/fixed (384d) | query Q5 (direct)
  question : How many API requests per minute can a Pro account make?
  gold doc : 07-architecture | context 5/5 | answer 0/5 (states a contradicting fact)
  retrieved:
    *rank 1 score 0.6245 [07-architecture] 100w
       Pro accounts are limited to sixty requests per minute and Team accounts to six hundred requests per minute, measured per account rather than per token. Exceeding the limit returns HTTP four twenty nine with a Retry After header. Outbound webhooks are retried up to five times with exponential backoff
     rank 2 score 0.5391 [06-support] 110w
       Support is delivered by email and by an in application contact form. The Free plan receives best effort support with no committed response time, and in practice most free tier questions are answered within three business days. The Pro plan commits to a first response within one business day. 

## Step 3.2 — Root cause analysis

The rubric separates retrieval from generation, so each failure can be attributed from its own
score pair rather than by inspection. The cell below counts the causes; the written diagnosis
is in `rag_analysis.md`.

| Score pattern | Cause | Why |
|---|---|---|
| ctx 0 | **embedding / chunking** | the answer never entered the prompt, so generation never had a chance |
| ctx 3 | **chunking** | the right document was retrieved but the chunk boundary separated the question's wording from the answer |
| ctx 5, ans 1 | **generation** | the answer was in the prompt and the model did not state it |
| ctx 5, ans 0 | **generation** | the model either abstained with the answer in front of it, or picked a competing figure out of the context |

**What the classifier cannot see.** It reads only the two scores, so it cannot tell query formulation from a generation error, and it trusts the grader. `rag_analysis.md` (Part 3) reads every failure's retrieved chunks and re-attributes the 10 rows: **4 generation** (wrong figure, merged facts, one format slip), **3 query formulation** (H6, a symptom question with two valid explanations in the corpus), **2 chunking** (Q1, Q2) and **1 embedding** (H7). Reading the chunks also exposed one grader false positive. For H7 under medium + hybrid, the accepted answer token "five" matched "five *business days*" in the support document, so a retrieval miss was scored context 5. Corrected, that configuration is context 4.64 and MRR 0.714.

In [23]:
cause_counts = {}
for f in failures:
    k, why = classify(f)
    cause_counts.setdefault((k, why), 0)
    cause_counts[(k, why)] += 1

print(f"{'cause':<12} {'n':>3}  explanation")
print("-" * 92)
for (k, why), n in sorted(cause_counts.items(), key=lambda kv: -kv[1]):
    print(f"{k:<12} {n:>3}  {why}")

tot = sum(cause_counts.values())
retr = sum(n for (k, _), n in cause_counts.items() if k in ("RETRIEVAL", "CHUNKING"))
print(f"\nretrieval-side {retr}/{tot} | generation-side {tot - retr}/{tot}")
print("\nfailures per query (which questions are intrinsically hard):")
perq = {}
for f in failures:
    perq[f["query_id"]] = perq.get(f["query_id"], 0) + 1
for qid, n in sorted(perq.items(), key=lambda kv: -kv[1]):
    q = next(x for x in rk.QUERIES if x["id"] == qid)
    print(f"  {qid} ({q['difficulty']:<6}) failed in {n}/9 configs — {q['text']}")

cause          n  explanation
--------------------------------------------------------------------------------------------
GENERATION     5  the answer was in the context and the model did not state it
GENERATION     2  the answer was in the context and the model abstained anyway
GENERATION     1  the model answered with a competing fact from the context
RETRIEVAL      1  the document holding the answer was never retrieved
CHUNKING       1  the right document was retrieved, but not the chunk with the answer

retrieval-side 2/10 | generation-side 8/10

failures per query (which questions are intrinsically hard):
  H6 (hard  ) failed in 3/9 configs — Why would my notifications go silent overnight?
  Q7 (direct) failed in 2/9 configs — How long is the refund window on a paid plan?
  Q5 (direct) failed in 1/9 configs — How many API requests per minute can a Pro account make?
  Q2 (direct) failed in 1/9 configs — How long are notifications kept on the free plan?
  Q1 (direct) failed in 1/9 

## Step 3.3 — System improvement

The most-failing configuration is fixed two ways and re-measured on the identical query set:

1. **Raise top-k from 3 to 5.** If the answer was ranked 4th or 5th it now enters the prompt.
   Costs prompt tokens, which Part 4.1 quantifies.
2. **Swap the chunking strategy**, keeping the embedding model fixed, which isolates the
   effect of chunking alone. The replacement is chosen from the grid as the best-scoring
   strategy the target is not already using.

Both are re-run live, so the before/after comparison uses the same grader and the same seed.

In [24]:
target = min(grid, key=lambda c: (c["ans"], c["mrr"]))
print(f"targeting the weakest configuration: {target['embed_size']} ({target['dim']}d) + "
      f"{target['strategy']}  ->  ctx {target['ctx']:.2f} ans {target['ans']:.2f} "
      f"mrr {target['mrr']:.3f}\n")

def summarise(rows):
    return {"ctx": st.mean(r["context_quality"] for r in rows),
            "ans": st.mean(r["answer_quality"] for r in rows),
            "mrr": st.mean((1 / r["answer_rank"]) if r["answer_rank"] else 0 for r in rows),
            "full": sum(1 for r in rows if r["answer_quality"] == 5),
            "ctx_words": st.mean(sum(h["words"] for h in r["retrieved"]) for r in rows)}

# The alternative strategy has to be one the target is NOT already using. Hard-coding
# "hybrid" here was a bug: the weakest configuration turned out to be medium/hybrid, so
# "switch to hybrid" was a no-op that produced byte-identical numbers and looked like a fix
# that achieved nothing. Pick the best-scoring OTHER strategy for the same embedding model.
alt_strat = max((c for c in grid if c["embed_size"] == target["embed_size"]
                 and c["strategy"] != target["strategy"]),
                key=lambda c: (c["ans"], c["mrr"]))["strategy"]
print(f"alternative chunking strategy for {target['embed_size']}: {alt_strat} "
      f"(best of the other two for this embedding model)\n")

variants = {f"before (k={K}, {target['strategy']})": target["rows"]}
variants["fix 1: k=5"] = rk.run_config(stores[(target["embed_size"], target["strategy"])], k=5)
variants[f"fix 2: {alt_strat} chunking, k={K}"] = rk.run_config(
    stores[(target["embed_size"], alt_strat)], k=K)

print(f"{'variant':<34} {'ctx':>6} {'ans':>6} {'MRR':>7} {'5/5 answers':>12} {'ctx words':>10}")
print("-" * 82)
for name, rows in variants.items():
    s = summarise(rows)
    print(f"{name:<34} {s['ctx']:>6.2f} {s['ans']:>6.2f} {s['mrr']:>7.3f} "
          f"{s['full']:>8}/{len(rows):<3} {s['ctx_words']:>10.0f}")

targeting the weakest configuration: medium (768d) + fixed  ->  ctx 4.86 ans 4.36 mrr 0.821

alternative chunking strategy for medium: overlap (best of the other two for this embedding model)



variant                               ctx    ans     MRR  5/5 answers  ctx words
----------------------------------------------------------------------------------
before (k=3, fixed)                  4.86   4.36   0.821       12/14         292
fix 1: k=5                           5.00   5.00   0.839       14/14         483
fix 2: overlap chunking, k=3         5.00   5.00   0.881       14/14         577


In [25]:
# Query-by-query, so the change is attributable rather than just an average moving.
base = {r["query_id"]: r for r in target["rows"]}
print(f"{'q':<4} {'set':<7} {'before ctx/ans':>15} {'k=5 ctx/ans':>13} "
      f"{alt_strat + ' ctx/ans':>15}  changed")
print("-" * 80)
for q in rk.QUERIES:
    b = base[q["id"]]
    k5 = next(r for r in variants["fix 1: k=5"] if r["query_id"] == q["id"])
    hy = next(r for r in variants[f"fix 2: {alt_strat} chunking, k={K}"] if r["query_id"] == q["id"])
    changed = []
    if (k5["context_quality"], k5["answer_quality"]) != (b["context_quality"], b["answer_quality"]):
        changed.append("k=5")
    if (hy["context_quality"], hy["answer_quality"]) != (b["context_quality"], b["answer_quality"]):
        changed.append(alt_strat)
    print(f"{q['id']:<4} {q['difficulty']:<7} "
          f"{b['context_quality']:>8}/{b['answer_quality']:<6} "
          f"{k5['context_quality']:>6}/{k5['answer_quality']:<6} "
          f"{hy['context_quality']:>8}/{hy['answer_quality']:<6}  {', '.join(changed) or '-'}")

q    set      before ctx/ans   k=5 ctx/ans overlap ctx/ans  changed
--------------------------------------------------------------------------------
Q1   direct         3/0           5/5             5/5       k=5, overlap
Q2   direct         5/5           5/5             5/5       -
Q3   direct         5/5           5/5             5/5       -
Q4   direct         5/5           5/5             5/5       -
Q5   direct         5/5           5/5             5/5       -
Q6   direct         5/5           5/5             5/5       -
Q7   direct         5/1           5/5             5/5       k=5, overlap
H1   hard           5/5           5/5             5/5       -
H2   hard           5/5           5/5             5/5       -
H3   hard           5/5           5/5             5/5       -
H4   hard           5/5           5/5             5/5       -
H5   hard           5/5           5/5             5/5       -
H6   hard           5/5           5/5             5/5       -
H7   hard           5/5

---
# Part 4 — System design reflection

## Step 4.1 — Cost awareness

Three knobs drive cost, and they do not cost the same thing. The cell below measures each from
this system rather than estimating:

* **Embedding size** — a one-off index cost in memory and encode time. Scales linearly in
  dimensions and in chunk count: `bytes = n_chunks × dim × 4`.
* **Chunk size** — sets how many chunks exist *and* how many words each one drags into the
  prompt when retrieved.
* **top-k** — the dominant *recurring* cost. Every retrieved chunk is pasted into the prompt on
  every single query, so k multiplies the input tokens of every request forever, while the
  embedding model is paid for once at index time.

In [26]:
print("INDEX COST (one-off, per embedding model)")
print(f"{'embedding':<8} {'dim':>5} {'strategy':<9} {'chunks':>7} {'index KB':>10} {'encode s':>9}")
print("-" * 58)
for c in grid:
    kb = c["n_chunks"] * c["dim"] * 4 / 1024
    print(f"{c['embed_size']:<8} {c['dim']:>5} {c['strategy']:<9} {c['n_chunks']:>7} "
          f"{kb:>10.1f} {c['index_sec']:>9.2f}")

print("\nPROMPT COST (recurring, charged on every query)")
print(f"{'strategy':<9} {'k':>2} {'ctx words':>10} {'prompt tokens':>14} {'output tokens':>14}")
print("-" * 54)
for strat in ("fixed", "overlap", "hybrid"):
    c = agg("strategy")[strat][0]
    pt = [r["prompt_tokens"] for r in c["rows"] if r["prompt_tokens"]]
    ot = [r["gen_eval_count"] for r in c["rows"] if r["gen_eval_count"]]
    cw = st.mean(sum(h["words"] for h in r["retrieved"]) for r in c["rows"])
    print(f"{strat:<9} {K:>2} {cw:>10.0f} {st.mean(pt):>14.0f} {st.mean(ot):>14.0f}")

# What does raising top-k actually cost? Measured, using the k=5 variant from Step 3.3.
k3 = [r["prompt_tokens"] for r in variants[list(variants)[0]] if r["prompt_tokens"]]
k5 = [r["prompt_tokens"] for r in variants["fix 1: k=5"] if r["prompt_tokens"]]
print(f"\ntop-k=3 -> {st.mean(k3):.0f} prompt tokens/query")
print(f"top-k=5 -> {st.mean(k5):.0f} prompt tokens/query "
      f"({st.mean(k5)/st.mean(k3)-1:+.0%})")
print(f"\nAt 10,000 queries/day that difference is "
      f"{(st.mean(k5)-st.mean(k3))*10000/1e6:.2f}M extra input tokens per day.")
print(f"The largest index here is only "
      f"{max(c['n_chunks']*c['dim']*4/1024 for c in grid):.0f} KB and is paid once.")

INDEX COST (one-off, per embedding model)
embedding   dim strategy   chunks   index KB  encode s
----------------------------------------------------------
small      384 fixed          32       48.0      1.48
small      384 overlap        24       36.0      1.11
small      384 hybrid         21       31.5      0.94
medium     768 fixed          32       96.0     13.95
medium     768 overlap        24       72.0     14.42
medium     768 hybrid         21       63.0      9.46
large     1024 fixed          32      128.0     21.04
large     1024 overlap        24       96.0     27.36
large     1024 hybrid         21       84.0     28.84

PROMPT COST (recurring, charged on every query)
strategy   k  ctx words  prompt tokens  output tokens
------------------------------------------------------
fixed      3        294            510             24
overlap    3        578            860             32
hybrid     3        417            727             30

top-k=3 -> 508 prompt tokens/query
to

### Which factor matters most

Ranked by how much each one moves the cost of a request at steady state:

1. **Prompt size, which top-k and chunk size multiply together.** Both are paid on every
   request:
   * Raising k from 3 to 5 added 51% prompt tokens (measured above).
   * Overlapping chunks carry 70% more prompt tokens than fixed-length chunks at the same k, and
     took 51% longer to generate on this CPU (14.4 s vs 9.5 s), because a CPU model reads every
   prompt token before it writes one.

   On a hosted API this is the bill; on our own hardware it is the latency.
2. **Embedding size.** This is not a one-off cost, because every query is embedded too. With
   the generator idle, one query embeds in 25 ms with the 384-d model and 329 ms with the
   1024-d one, 13× more on every request. The index itself is tiny (at most 128 KB here) and is
   built once.

   Embedding size ranks second only because 0.33 s is small next to ~12 s of generation. It
   still buys nothing here: Step 2.3 found no gain in answers from the larger models.

## Step 4.2 — RAG vs alternatives

| Approach | When to use | Evidence from our own work |
|---|---|---|
| **RAG** | Facts change often; answers must cite a source; the corpus is large relative to what a prompt can hold | Here RAG answered 4.36–5.00 out of 5 with ~500–860 prompt tokens per question, and editing a document changes the next answer with no retraining. |
| **Fine-tuning** | The *format*, tone or vocabulary must change; the task is stable; prompt cost must be minimised | In A7 Part 2 (notebook 3), fine-tuning our from-scratch character-level SLM was the only condition that got facts into answers (7.3/10 on Q1, where RAG and the baseline scored 2.0). But it also mixed them up: asked what membership costs, it answered "far dorms costs five dollars" (the answer is ten). And every change of fact means retraining. |
| **Pure prompting** | The whole knowledge fits comfortably in the context window and rarely changes | Tested on this corpus with Granite: the corpus is 4,184 tokens, and with Ollama's default context only 2,050 reached the model, so half was silently truncated. With the context raised to fit, one answer took 93.5 s against ~12 s with RAG, using 5–8× the prompt tokens. |

A7 Part 2's RAG failed only because that model's context window was 32 characters, so the
retrieved fact never fit; its conclusion was "RAG, on a model with a usable context window".
This assignment is that experiment, and RAG works. Even at 3,093 words, pure prompting needs its
context configured explicitly or it fails without an error, and it costs several times more per
question.

The approaches are complementary: fine-tune for *how* to answer, retrieve for *what is true*.
Step 2.5's rising noise shows where retrieval earns its place as the corpus grows.

## Step 4.3 — System design at 10,000 users/day

```
                          ┌──────────────────────────────────────────┐
                          │             INGESTION (offline)          │
                          │  documents → chunker → embedding model   │
                          │                    ↓                     │
                          │          vector store (upsert)           │
                          └──────────────────────────────────────────┘
                                             ↑ nightly / on edit
  ┌──────┐   query    ┌───────────┐  vector  ┌──────────────┐  top-k   ┌─────────────┐
  │ User │ ─────────► │   API     │ ───────► │ Vector store │ ───────► │   Prompt    │
  └──────┘            │  (REST)   │          │  (ANN index) │  chunks  │  assembly   │
      ▲               └───────────┘          └──────────────┘          └─────────────┘
      │                     │                                                │
      │                     ▼ cache lookup                                    ▼
      │              ┌─────────────┐                                   ┌─────────────┐
      │              │ query cache │                                   │     LLM     │
      │              │  (Redis)    │                                   │granite3.3:2b│
      │              └─────────────┘                                   └─────────────┘
      │                     │ hit                                             │
      └─────────────────────┴──────────────── response ◄──────────────────────┘
```

**Mermaid version of the same flow:**

```mermaid
flowchart LR
    U[User] --> Q[Query]
    Q --> C{Cache hit?}
    C -- yes --> R[Response]
    C -- no --> E[Embed query]
    E --> V[(Vector store)]
    V --> K[Top-k chunks]
    K --> P[Prompt assembly]
    P --> L[LLM granite3.3:2b]
    L --> R
    D[Documents] --> CH[Chunker] --> EM[Embedding model] --> V
```

### Scaling to 10,000 users/day

**Load.** We assume 3 questions per user per day, which gives 30,000 queries/day: 0.35/s on
average and about 1.7/s at a 5× daytime peak.

**Where the time goes, measured above.** Retrieval takes 25 ms with the small model. Generation
takes 6–26 s per answer on this laptop CPU (mean 12.2 s), one request at a time, so the peak
would need ~20 CPU generation workers. Generation is the component to engineer, not the vector
store.

**What we would optimise, in order**

1. **Serve generation from a GPU with batching (e.g. vLLM).** A6 measured Granite-3.3-2B at
   ~1.7 s per message on a single T4, and batching serves many requests at once.
2. **Cache answers**, keyed on the normalised query plus the retrieved chunk IDs, so a document
   edit invalidates only the answers that used it. A hit skips generation entirely.
3. **Keep the prompt small:** k=3 by default, raising k or switching to wider chunks only for
   the query types that need it (Steps 3.3 and 4.1).
4. **Keep the small embedding model, resident in memory.** It answered as well as the large one
   at 1/13 of the per-query cost.
5. **Move from exact NumPy search to pgvector or FAISS past ~10⁵ chunks.** Below that, exact
   search is faster and has no approximation error.
6. **Monitor retrieval noise**, which rises well before answer quality falls (Step 2.5).

**What we would not do:** buy a bigger embedding model. Part 3 put the failures in generation
and in the questions themselves, not in the embeddings.

---
## Summary of findings

The cells above produce the numbers; `rag_analysis.md` carries the full written analysis of this
same run. The results worth stating:

1. **A bigger embedding model bought slightly better retrieval, not better answers.** The 1024-d
   model never missed the right document (42/42), but answered 4.60 against 4.67 for the 384-d
   and 768-d models, at 13× the per-query embedding cost. Our first run, with a different
   generator, reached the same verdict.
2. **Answers tracked how much context each chunk carried:** fixed-length 4.45 < hybrid 4.69 <
   overlap 4.79. Raising k to 5 fixed the weakest configuration completely (14/14). More context
   costs prompt tokens and generation time on every request.
3. **Most failures were not retrieval failures.** Of 10 failing rows, 4 were generation errors
   with the answer in the prompt, 3 came from an ambiguous question, 2 from chunking and 1 from
   the embedding model. Reading the chunks also exposed one grader false positive.
4. **Noise leads, accuracy lags.** Off-document retrievals rose from 0% to 43% as the corpus
   grew from 2 to 8 documents, while answer quality stayed at 5.00 throughout.

In [27]:
# Write this run's results out, so rag_analysis.md can be generated from the numbers this
# notebook actually produced rather than from a separate run of the scripts.
import pathlib
pathlib.Path("results").mkdir(exist_ok=True)
payload = {
    "generation_model": rk.GEN_MODEL, "top_k": K,
    "corpus": {"documents": len(docs), "paragraphs": n_paras, "words": n_words},
    "embedders": {s: {"name": embedders[s]["name"], "dim": embedders[s]["dim"],
                      "load_sec": embedders[s]["load_sec"]} for s in embedders},
    "chunksets": {k2: {"n_chunks": len(v), "mean_words": st.mean(c["words"] for c in v)}
                  for k2, v in chunksets.items()},
    "grid": grid, "scaling": scaling, "scaling_config": [SCALE_SIZE, SCALE_STRAT],
    "failures": failures, "cause_counts": {f"{k2[0]}|{k2[1]}": v for k2, v in cause_counts.items()},
    "improvement": {"target": {"embed_size": target["embed_size"], "dim": target["dim"],
                               "strategy": target["strategy"]},
                    "variants": {name: summarise(rows) for name, rows in variants.items()}},
}
pathlib.Path("results/notebook_run.json").write_text(json.dumps(payload, indent=2, default=str))
print("wrote results/notebook_run.json")
print(f"  {len(grid)} configurations, {len(failures)} failing rows, "
      f"{len(scaling)} scaling points")

wrote results/notebook_run.json
  9 configurations, 10 failing rows, 4 scaling points
